# Beyond BLAST Notebook

In [ ]:
ENV["JULIA_PKG_PRECOMPILE_AUTO"] = 0;
using Pkg
Pkg.activate("blast_code"; io=devnull)
Pkg.resolve(; io=devnull)
Pkg.instantiate(; io=devnull)

using Revise 

using Base.Threads, NPZ, DataInterpolations, Interpolations, FastChebInterp
using BenchmarkTools, FFTW, FastTransforms, Dates, TOML, Plots, Plots.Measures
using QuadGK, LaTeXStrings, Tullio, StaticArrays, LoopVectorization, LinearAlgebra
using Unitful, SpecialFunctions, DifferentialEquations, Cosmology, NumericalIntegration
using CSV, DataFrames, JSON, OrderedCollections
using MCIntegration
;

In [ ]:
include("blast_code/src/Blast.jl")
include("blast_code/src/blast_tutorials.jl")
include("blast_code/src/galaxy_galaxy.jl")
include("blast_code/src/shear_shear.jl")
include("blast_code/src/config.jl")
include("blast_code/src/paths.jl")
include("blast_code/src/plot_config.jl")
using .galaxy_galaxy
using .shear_shear
using .Blast
using .blast_tutorials

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 1. COSMOLOGICAL FUNCTIONS (stubs — replace with your actual BLAST routines)
# ─────────────────────────────────────────────────────────────────────────────

# Comoving distance range supported by your redshift kernel n(χ).
# Tune these to match your survey's χ_min and χ_max in Mpc/h or Mpc.
const χ_min = 100.0   # Mpc/h
const χ_max = 3000.0  # Mpc/h

# Linear matter power spectrum P_lin(k) [Mpc/h]^3
# Replace with your interpolated Boltzmann code output (e.g. from CAMB/CLASS).
function P_lin(k::Float64)::Float64
    # Simple Harrison–Zel'dovich-like toy model for testing
    return k^(-3.0) * exp(-k^2 / 0.1)
end

# Hubble parameter H(χ) in units where c=1 (or keep c explicit below)
function H_over_c(χ::Float64)::Float64
    # H(z)/c in 1/Mpc, flat ΛCDM toy model
    return 1.0 / 3000.0
end

# Galaxy bias b(χ)
function b(χ::Float64)::Float64
    return 1.5
end

# Redshift-space galaxy number density kernel n(χ) [normalised]
function n_gal(χ::Float64)::Float64
    χ0 = 1500.0; σ = 300.0
    return exp(-0.5 * ((χ - χ0) / σ)^2)
end

# Linear growth factor D(χ), normalised to D(χ=0)=1
function D_growth(χ::Float64)::Float64
    return 1.0 / (1.0 + χ / 3000.0)   # crude matter-dominated toy
end

# Spherical Bessel function j_l(x) via SpecialFunctions
# j_l(x) = sqrt(π / 2x) * J_{l+1/2}(x)
function j_sph(l::Int, x::Float64)::Float64
    x == 0.0 && return (l == 0 ? 1.0 : 0.0)
    return sqrt(π / (2x)) * besselj(l + 0.5, x)
end

# ─────────────────────────────────────────────────────────────────────────────
# 2. THE INTEGRAND KERNEL W̃_l^g(k_p, k)  (integrand of the χ-integral)
#    W̃_l^g(k_p, k) = ∫ dχ  χ² H(χ)b(χ)n(χ)D(χ)/c  k_p j_l(kχ) j_l(k_p χ)
#    Here we evaluate the *inner* χ-kernel at a single point χ,
#    so the full integral is the sum over χ weighted by the Jacobian.
# ─────────────────────────────────────────────────────────────────────────────

@inline function W_kernel(l::Int, k_p::Float64, k::Float64, χ::Float64)::Float64
    return χ^2 * H_over_c(χ) * b(χ) * n_gal(χ) * D_growth(χ) * k_p *
           j_sph(l, k * χ) * j_sph(l, k_p * χ)
end

# ─────────────────────────────────────────────────────────────────────────────
# 3. MAIN INTEGRATION ROUTINE
#    Computes S_l^g(k1, k2) for given l, k1, k2.
# ─────────────────────────────────────────────────────────────────────────────

function compute_Slg(l::Int, k1::Float64, k2::Float64;
                     k_min::Float64  = 1e-4,   # lower cutoff for k [1/Mpc/h]
                     neval::Int      = 200_000,
                     niter::Int      = 20,
                     solver::Symbol  = :vegasmc,
                     verbose::Int    = 0)

    # ── 3a. Define the three integration variable pools ───────────────────────
    # Each Continuous pool will learn its own adaptive grid via Vegas.
    # We keep all three on [0, 1) and map to physical variables by hand.

    var_k  = Continuous(0.0, 1.0)   # mapped → k  ∈ [k_min, ∞)
    var_χ  = Continuous(0.0, 1.0)   # mapped → χ  ∈ [χ_min, χ_max]
    var_χp = Continuous(0.0, 1.0)   # mapped → χ' ∈ [χ_min, χ_max]

    # Pack them into a CompositeVar so that Vegas treats each pool separately.
    # Inside the integrand they are unpacked as (yk, yχ, yχp).
    vars = CompositeVar(var_k, var_χ, var_χp)

    # Precompute the χ-interval width (Jacobian factor for the linear map)
    Δχ = χ_max - χ_min

    # ── 3b. Define the integrand ──────────────────────────────────────────────
    function integrand((yk, yχ, yχp), config)

        # --- Map y ∈ [0,1) to k ∈ [k_min, ∞) using x = a + y/(1-y)
        # Jacobian: dk/dy = 1/(1-y)^2
        yk1  = yk[1]
        k    = k_min + yk1 / (1.0 - yk1)
        Jk   = 1.0 / (1.0 - yk1)^2        # Jacobian from the change of variables

        # --- Map y ∈ [0,1) to χ ∈ [χ_min, χ_max] (simple linear rescaling)
        # Jacobian: dχ/dy = Δχ
        χ    = χ_min + yχ[1]  * Δχ
        χp   = χ_min + yχp[1] * Δχ
        Jχ   = Δχ              # same Jacobian for both χ and χ'

        # --- Evaluate the kernel at the sampled (k, χ, χ') point
        # k^2 P_lin(k)  ×  W_kernel(k1, k, χ)  ×  W_kernel(k2, k, χ')
        # then multiply by the three Jacobians
        fk  = k^2 * P_lin(k)
        fχ  = W_kernel(l, k1, k, χ)
        fχp = W_kernel(l, k2, k, χp)

        return fk * fχ * fχp * Jk * Jχ * Jχ  # Jχ appears twice, one per χ integral
    end

    # ── 3c. Run the integration ───────────────────────────────────────────────
    # dof = [1,1,1] means: 1 variable from var_k, 1 from var_χ, 1 from var_χp
    result = integrate(integrand;
                       var     = vars,
                       dof     = [[1, 1, 1]],
                       solver  = solver,
                       neval   = neval,
                       niter   = niter,
                       verbose = verbose)

    # ── 3d. Extract result ────────────────────────────────────────────────────
    Nlg = 1.0  # replace with your actual normalisation N_l^g

    val  = Nlg * result.mean[1]
    err  = Nlg * result.stdev[1]
    chi2 = result.chi2[1]

    return val, err, chi2
end

# ─────────────────────────────────────────────────────────────────────────────
# 4. COMPUTING S_l^g ON A (k1, k2) GRID
#    This is the typical use-case: you want the full matrix S[i,j] = S_l^g(k1_i, k2_j).
#    Because S is symmetric in k1 ↔ k2, compute only the upper triangle.
# ─────────────────────────────────────────────────────────────────────────────

function compute_Slg_matrix(l::Int, k_grid::Vector{Float64}; kwargs...)
    N = length(k_grid)
    S_mean  = zeros(N, N)
    S_err   = zeros(N, N)

    for i in 1:N
        for j in i:N   # upper triangle only (symmetry)
            val, err, _ = compute_Slg(l, k_grid[i], k_grid[j]; kwargs...)
            S_mean[i, j] = val
            S_mean[j, i] = val   # fill lower triangle by symmetry
            S_err[i, j]  = err
            S_err[j, i]  = err
            @info "S[$i,$j] = $val ± $err"
        end
    end
    return S_mean, S_err
end

# ─────────────────────────────────────────────────────────────────────────────
# 5. EXAMPLE RUN
# ─────────────────────────────────────────────────────────────────────────────

l  = 10
k1 = 0.05   # 1/Mpc/h
k2 = 0.10

val, err, chi2 = compute_Slg(l, k1, k2; neval=300_000, niter=20, verbose=1)
println("S_$(l)^g($k1, $k2) = $val  ±  $err   (χ²/dof = $chi2)")
